# 02 - Data Cleaning & Derived Variables (Làm sạch dữ liệu & Tạo biến phái sinh)

**Mục đích:** 
- Hiện thực hóa các quyết định xử lý dữ liệu đã quan sát ở Notebook 01.
- Tạo các biến phái sinh quan trọng: `IsCancelled`, `Revenue`, `InvoiceValue`.
- Xuất 2 bảng dữ liệu sạch vào `data/processed/` phục vụ cho các Phase phân tích tiếp theo:
  1. `retail_cleaned.csv` (cấp độ dòng giao dịch)
  2. `invoice_data.csv` (cấp độ hóa đơn)

## 0. Chuẩn bị thư viện và nạp module

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Tự động nhận diện thư mục gốc của project
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_raw_data
from src.data.cleaning import clean_retail_data, create_invoice_level_data, save_processed_data

# Hiển thị số thực với 2 chữ số thập phân
pd.options.display.float_format = "{:.2f}".format

## 1. Nạp dữ liệu thô (Raw Data)

In [ ]:
raw_df = load_raw_data()
print("Kích thước dữ liệu gốc:", raw_df.shape)
raw_df.head()

## 2. Tiến hành làm sạch dữ liệu giao dịch

Quy trình xử lý theo đúng nguyên tắc nghiệp vụ:
1. **Xóa dòng trùng hoàn toàn (drop_duplicates):** Loại bỏ lỗi ghi nhận đúp.
2. **Tạo biến `IsCancelled`:** Đơn hủy bắt đầu bằng chữ cái `C`. Không xóa đơn hủy mà gắn cờ để kiểm định ở Phase 6.
3. **Lọc bỏ giao dịch bất thường:** Lọc bỏ `UnitPrice <= 0` (hàng tặng/nợ xấu) và `Quantity < 0` không phải đơn hủy (kiểm kê kho).
4. **Tạo biến doanh thu:** `Revenue = Quantity * UnitPrice`.

In [ ]:
df_clean = clean_retail_data(raw_df)
print("\nKích thước sau làm sạch:", df_clean.shape)
df_clean.head()

## 3. Kiểm tra tính toàn vẹn của dữ liệu sau làm sạch

In [ ]:
print("--- Phân bố đơn hủy (IsCancelled) ---")
print(df_clean["IsCancelled"].value_counts())
print()

print("--- Kiểm tra giá trị Doanh thu (Revenue) ---")
print("Số dòng doanh thu âm (đơn hủy hoàn tiền):", (df_clean["Revenue"] < 0).sum())
print("Số dòng doanh thu dương (bán hàng thành công):", (df_clean["Revenue"] > 0).sum())
print()

# Thống kê mô tả 3 biến số lượng, đơn giá và doanh thu
df_clean[["Quantity", "UnitPrice", "Revenue"]].describe()

## 4. Tổng hợp dữ liệu cấp độ Hóa đơn (Invoice Level)

> **Phân biệt rõ:**
> - `Revenue`: Giá trị bán hàng ở mức từng dòng sản phẩm.
> - `InvoiceValue`: Tổng giá trị tiền tệ của một hóa đơn (`SUM(Revenue)`).

In [ ]:
df_invoice = create_invoice_level_data(df_clean)
print("Kích thước bảng hóa đơn:", df_invoice.shape)
df_invoice.head()

## 5. Lưu kết quả ra thư mục `data/processed/`

In [ ]:
save_processed_data(df_clean, df_invoice)
print("Lưu file hoàn tất!")